# AudiQ AI v0.1 — Symbol Detection

Goal: train a first computer-vision model to locate audiogram symbols. This notebook does **not** make autonomous clinical decisions. Extracted thresholds must be visually verified before AudiQ calculates PTA or interpretation.


In [ ]:
!pip -q install ultralytics pyyaml


In [ ]:
from pathlib import Path
import yaml
DATA = Path('/kaggle/input/audiq-dataset/dataset.yaml')
assert DATA.exists(), f'Missing {DATA}. Attach your permitted YOLO dataset first.'
print(DATA)
print(yaml.safe_load(DATA.read_text()))


In [ ]:
# Validate the dataset before spending GPU time.
# If ai/prepare_dataset.py is available in your working copy:
# !python ai/prepare_dataset.py --root /kaggle/input/audiq-dataset/dataset


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data=str(DATA),
    epochs=50,
    imgsz=960,
    batch=-1,
    patience=15,
    pretrained=True,
    project='/kaggle/working/audiq_runs',
    name='symbol_v0_1',
)


In [ ]:
metrics = model.val(data=str(DATA), imgsz=960)
print(metrics)


In [ ]:
# Test predictions on held-out images.
TEST_DIR = '/kaggle/input/audiq-dataset/dataset/images/test'
model.predict(source=TEST_DIR, imgsz=960, conf=0.25, save=True, save_txt=True, project='/kaggle/working/audiq_predictions', name='test')


In [ ]:
# Export only after validation.
model.export(format='onnx')


## Acceptance gate

Before integrating a model into AudiQ, record precision/recall/mAP, threshold error in dB, frequency-assignment accuracy, confidence calibration, and manual-correction rate on a held-out set. A low-confidence prediction must remain editable/reviewable. Do not place patient images or restricted datasets in the public repository.
